# GPU Final Train Notebook - Units SAEs

This notebook retrains unit-specific SAEs for all selected layers (4, 8, 12, 16, 20, 24, 28), saves them in a separate `final_train` checkpoint folder, rebuilds the unit contrast graphs, and reruns only the affected unit interventions.

The previous Drive checkpoints remain untouched. This run is meant to test whether the weak sparse SAE interventions were caused by using mixed/general SAEs rather than unit-specialized SAEs across the selected layers.


## Step 1 - Mount Drive and fetch code

GitHub is preferred for code. Drive `project.zip` remains a fallback. SAE checkpoints and tuned outputs persist in Drive.


In [ ]:
# Step 1: Mount Google Drive and fetch project code
# GitHub is the primary code source. Drive project.zip remains a backup.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = "https://github.com/evey-dev/test_run.git"
repo_dir = "/content/test_run"

def run_cmd(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

github_ok = False
try:
    clone_dir = repo_dir
    if os.path.isdir(os.path.join(clone_dir, ".git")):
        run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
    else:
        if os.path.exists(clone_dir) and os.listdir(clone_dir):
            clone_dir = "/content/test_run_github"
        if os.path.isdir(os.path.join(clone_dir, ".git")):
            run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
        elif os.path.exists(clone_dir) and os.listdir(clone_dir):
            raise RuntimeError(f"{clone_dir} exists but is not a git repo")
        else:
            run_cmd(["git", "clone", "--depth", "1", repo_url, clone_dir])
    os.chdir(clone_dir)
    github_ok = True
    print(f"Using GitHub checkout: {os.getcwd()}")
except Exception as exc:
    print("GitHub checkout failed; falling back to Drive project.zip.")
    print(repr(exc))

if not github_ok:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Could not find {zip_path}. Fix GitHub access or upload project.zip.")
    print(f"Extracting backup zip {zip_path}...")
    run_cmd(["unzip", "-q", "-o", zip_path, "-d", "/content/"])
    for candidate in ["/content/test_run", "/content/mphil_project/test_run", "/content"]:
        if os.path.isdir(os.path.join(candidate, "src")) and os.path.isdir(os.path.join(candidate, "configs")):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError("Could not find extracted project root containing src/ and configs/.")
    print(f"Using Drive backup project: {os.getcwd()}")

print(f"Current working directory: {os.getcwd()}")
!ls -l


## Step 2 - Install dependencies


In [ ]:
# Step 2: Install dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .


## Step 3 - Generate units data


In [ ]:
# Step 3: Generate datasets
!python data/generate_datasets.py
print("Dataset files:")
!ls -lh data/units_data.csv


## Step 4 - Define final-run folders

The final checkpoints use a new Drive folder, so this will not overwrite or silently resume from the earlier `sae_checkpoints_units_l24_l28_hyper` run.


In [ ]:
# Final run folders
from pathlib import Path

FINAL_LAYERS = [4, 8, 12, 16, 20, 24, 28]
FINAL_DATA_DIR = 'mechanistic_data_units_final_train'
FINAL_SAE_DIR = 'mechanistic_data/sae_checkpoints_units_final_train'
FINAL_DRIVE_SAE_DIR = '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_final_train'
FINAL_CONFIG = 'configs/sae_units_final_train_config.yaml'

Path('configs').mkdir(exist_ok=True)
print('Final layers:', FINAL_LAYERS)
print('Activation dir:', FINAL_DATA_DIR)
print('Local SAE dir:', FINAL_SAE_DIR)
print('Drive SAE dir:', FINAL_DRIVE_SAE_DIR)
print('Config:', FINAL_CONFIG)


## Step 5 - Capture units-only activations for all selected layers


In [ ]:
# Capture units-only MLP activations for final SAE training
!python src/capture_activations.py \
  --output-dir mechanistic_data_units_final_train \
  --behaviours units \
  --layers 4 8 12 16 20 24 28 \
  --seed 787

print("\nFinal units-only activation bundle:")
!ls -lh mechanistic_data_units_final_train


## Step 6 - Train final unit-specific SAEs for all selected layers


In [ ]:
# Train final unit-specific SAEs for all selected layers
# The config is written in the runtime too, so this notebook works even before a GitHub push.
from pathlib import Path

Path('configs/sae_units_final_train_config.yaml').write_text('''output_dir: "mechanistic_data/sae_checkpoints_units_final_train"
data_dir: "mechanistic_data_units_final_train"
layers:
  - 4
  - 8
  - 12
  - 16
  - 20
  - 24
  - 28
hidden_size: 2560
latent_dim: 8192
batch_size: 64
epochs: 75
lr: 0.001
l1_lambda: 0.0003
seed: 787
device: "auto"
''')
print('Wrote configs/sae_units_final_train_config.yaml')

!python src/train.py \
  --config configs/sae_units_final_train_config.yaml \
  --drive-dir /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_final_train

print("\nFinal local checkpoints:")
!ls -lh mechanistic_data/sae_checkpoints_units_final_train
print("\nFinal Drive checkpoints:")
!ls -lh /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_final_train


## Step 7 - Verify final checkpoint set


In [ ]:
# Verify final checkpoint directory before running expensive attribution graphs
from pathlib import Path

layers = [4, 8, 12, 16, 20, 24, 28]
ckpt_dir = Path('mechanistic_data/sae_checkpoints_units_final_train')
missing = []
for layer in layers:
    for name in [f'sae_layer{layer}.pt', f'sae_layer{layer}_metadata.json', f'latents_layer{layer}.npy']:
        if not (ckpt_dir / name).exists():
            missing.append(str(ckpt_dir / name))

if missing:
    raise FileNotFoundError('Missing final SAE artifacts:\n' + '\n'.join(missing))

print('All final SAE artifacts found.')
print('Final SAE config:')
!cat configs/sae_units_final_train_config.yaml
print('Checkpoint directory:')
!ls -lh mechanistic_data/sae_checkpoints_units_final_train


## Step 8 - Rebuild units contrast graphs with final unit-specific SAEs


In [ ]:
# Final graph 1: force -> newtons, contrast joules
!python src/attribution_graph.py \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target "newtons" \
  --contrast-target "joules" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/units_final_force_graph.json \
  --output-html outputs/units_final_force_graph.html \
  --output-mermaid outputs/units_final_force_graph.md


In [ ]:
# Final graph 2: energy -> joules, contrast newtons
!python src/attribution_graph.py \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --target "joules" \
  --contrast-target "newtons" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/units_final_energy_graph.json \
  --output-html outputs/units_final_energy_graph.html \
  --output-mermaid outputs/units_final_energy_graph.md


## Step 9 - Rerun affected units interventions with final unit-specific SAEs


In [ ]:
# Final full latent swap: ENERGY -> FORCE
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --target-token "newtons, joules, volts" \
  --positions all \
  --layer-scan \
  --output outputs/units_final_swap_minpair_energy_to_force.json


In [ ]:
# Final full latent swap: FORCE -> ENERGY
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --target-token "newtons, joules, volts" \
  --positions all \
  --layer-scan \
  --output outputs/units_final_swap_minpair_force_to_energy.json


In [ ]:
# Final sparse swap: ENERGY graph-positive features -> FORCE run
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --graph-json outputs/units_final_energy_graph.json \
  --graph-feature-sign positive \
  --target-token "newtons, joules, volts" \
  --positions all \
  --output outputs/units_final_swap_sparse_energy_to_force.json


In [ ]:
# Final sparse swap: FORCE graph-positive features -> ENERGY run
!python src/intervention.py \
  --mode swap \
  --source-prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --prompt 'Fact: The official SI unit for the energy of a moving engine thrust is named "' \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --graph-json outputs/units_final_force_graph.json \
  --graph-feature-sign positive \
  --target-token "newtons, joules, volts" \
  --positions all \
  --output outputs/units_final_swap_sparse_force_to_energy.json


In [ ]:
# Final sparse inhibition: remove force-graph positive features from FORCE run
!python src/intervention.py \
  --mode inhibit \
  --prompt 'Fact: The official SI unit for the force of a moving engine thrust is named "' \
  --target-token "newtons, joules, volts" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_units_final_train_config.yaml \
  --graph-json outputs/units_final_force_graph.json \
  --graph-feature-sign positive \
  --positions last \
  --output outputs/units_final_inhibit_force.json


## Step 10 - Persist final-train outputs to Drive


In [ ]:
# Copy final-train graphs/results to Drive
import os, shutil, glob

drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/test_run/outputs/units_final_*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')
